# Setup

##  Imports

In [1]:
import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import torch
import torchaudio
from concurrent.futures import ThreadPoolExecutor
import tempfile

# Prevent torchaudio from breaking
if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

from pyannote.audio import Pipeline
from pydub import AudioSegment

W0425 15:49:07.873000 43244 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## Environment & API Setup

In [2]:
load_dotenv()
api_key = os.getenv("AQUEDUCT_API_KEY", os.getenv("API_KEY"))
hf_token = os.getenv("HF_TOKEN")

if not api_key or not hf_token:
    raise ValueError("Missing API Keys. Ensure AQUEDUCT_API_KEY and HF_TOKEN are set in .env")

client = OpenAI(api_key=api_key, base_url="https://aqueduct.ai.datalab.tuwien.ac.at/v1")

## Local Model

In [3]:
print("Loading Pyannote Diarization Model locally...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=hf_token
).to(device)

Loading Pyannote Diarization Model locally...


The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


## Path Configuration

In [4]:
BASE_FOLDER = Path("D:/TUW-BT/user_study_recordings")

## Helper Functions

In [5]:
def transcribe_full_audio(file_path: Path) -> dict:
    """Compresses the audio to a temporary file and sends it to Whisper."""
    print("      [API] Compressing audio to meet server upload limits...")

    # 1. Load and compress the audio
    audio = AudioSegment.from_file(str(file_path))
    # Downsampling to whisper's natively used 16kHz mono
    audio = audio.set_frame_rate(16000).set_channels(1)

    # 2. Create a temporary file path
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as temp_audio_file:
        temp_path = temp_audio_file.name

    try:
        # Export as a highly compressed MP3
        audio.export(temp_path, format="mp3", bitrate="32k")

        print("      [API] Starting full audio transcription...")
        with open(temp_path, "rb") as f:
            response = client.audio.transcriptions.create(
                model="whisper-large",
                file=f,
                response_format="verbose_json",
                timestamp_granularities=["segment"],
                language="en"
            )
        print("      [API] Transcription complete.")

    finally:
        # Clean up the temporary file
        if os.path.exists(temp_path):
            os.remove(temp_path)

    if hasattr(response, 'model_dump'):
        return response.model_dump()
    return response

def diarize_audio_locally(file_path: Path):
    """Runs Pyannote diarization on the full audio file locally."""
    print(f"      [Local] Starting diarization on {device}...")
    full_audio = AudioSegment.from_file(str(file_path))
    mono_audio = full_audio.set_channels(1)

    samples = np.array(mono_audio.get_array_of_samples())
    max_val = float(1 << (8 * mono_audio.sample_width - 1))
    samples = samples.astype(np.float32) / max_val
    waveform = torch.from_numpy(samples).unsqueeze(0)

    audio_in_memory = {
        "waveform": waveform,
        "sample_rate": mono_audio.frame_rate
    }

    diarization = diarization_pipeline(audio_in_memory)
    print("      [Local] Diarization complete.")
    return diarization

def align_segments(whisper_data: dict, diarization) -> str:
    """Maps Whisper text segments to Pyannote speakers based on timestamp overlap."""
    print("  -> Aligning timestamps...")
    segments = whisper_data.get("segments", [])
    raw_transcript = []

    # Extract the actual Annotation object from the DiarizeOutput wrapper
    annotation = diarization.speaker_diarization if hasattr(diarization, "speaker_diarization") else diarization

    for segment in segments:
        seg_start = segment["start"]
        seg_end = segment["end"]
        seg_text = segment["text"].strip()

        # Calculate which speaker was active the longest during this text segment
        overlap_durations = {}
        for turn, _, speaker in annotation.itertracks(yield_label=True):
            overlap_start = max(seg_start, turn.start)
            overlap_end = min(seg_end, turn.end)
            overlap = max(0, overlap_end - overlap_start)

            if overlap > 0:
                overlap_durations[speaker] = overlap_durations.get(speaker, 0) + overlap

        # Default to UNKNOWN if no speaker track overlaps
        dominant_speaker = "UNKNOWN"
        if overlap_durations:
            dominant_speaker = max(overlap_durations, key=overlap_durations.get)

        raw_transcript.append(f"[{seg_start:.1f}s - {seg_end:.1f}s] {dominant_speaker}: {seg_text}")

    return "\n".join(raw_transcript)

def clean_transcript_with_llm(raw_transcript: str) -> str:
    """Uses the LLM to fix speaker errors in chunks, using a context overlap for split sentences."""
    print("  -> Sending raw transcript to LLM in chunks for contextual cleanup...")

    # Explicitly instruct the LLM to remove timestamps and how to handle the context block
    system_prompt = (
        "You are an expert qualitative data assistant processing a chunk of a user study transcript. "
        "The input format is '[start - end] SPEAKER: text'. "
        "Your strict task:\n"
        "1. Identify who is the 'Instructor' and who is the 'Student' based on the context.\n"
        "2. Fix any obvious speaker misassignments.\n"
        "3. STRIP ALL TIMESTAMPS. Your output must strictly look like 'Instructor: [text]' or 'Student: [text]'. Do not include any time brackets.\n"
        "4. If a 'CONTEXT FROM PREVIOUS CHUNK' block is provided, read it to understand if the current chunk starts mid-sentence. DO NOT output the context text. If the current chunk continues a thought from the context, seamlessly merge it.\n"
        "5. DO NOT summarize, hallucinate, or remove any spoken words. Preserve the exact dialogue."
    )

    lines = raw_transcript.split('\n')
    chunk_size = 80
    final_cleaned_chunks = []

    total_chunks = (len(lines) + chunk_size - 1) // chunk_size
    previous_context = ""

    for i in range(0, len(lines), chunk_size):
        chunk_lines = lines[i:i + chunk_size]
        chunk_text = '\n'.join(chunk_lines)

        # Inject the last two lines of the previous chunk as read-only context
        if previous_context:
            input_payload = (
                f"--- CONTEXT FROM PREVIOUS CHUNK (DO NOT OUTPUT THIS) ---\n"
                f"{previous_context}\n"
                f"--- END CONTEXT ---\n\n"
                f"--- CURRENT CHUNK TO PROCESS ---\n"
                f"{chunk_text}"
            )
        else:
            input_payload = f"--- CURRENT CHUNK TO PROCESS ---\n{chunk_text}"

        current_chunk_num = (i // chunk_size) + 1
        print(f"      [LLM] Processing chunk {current_chunk_num}/{total_chunks}...")

        try:
            response = client.chat.completions.create(
                model="qwen-3.5-397b",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": input_payload}
                ],
                temperature=0.1
            )

            content = response.choices[0].message.content

            if content is None:
                finish_reason = response.choices[0].finish_reason
                print(f"      [LLM Warning] API returned empty content for chunk {current_chunk_num}. Reason: {finish_reason}")
                final_cleaned_chunks.append("\n[LLM RETURNED EMPTY RESPONSE FOR THIS CHUNK]\n" + chunk_text)
            else:
                cleaned_text = content.strip()
                final_cleaned_chunks.append(cleaned_text)

        except Exception as e:
            print(f"      [LLM Error] Failed to process chunk {current_chunk_num}: {e}")
            final_cleaned_chunks.append("\n[LLM PROCESSING ERROR FOR THIS CHUNK]\n" + chunk_text)

        # Grab the last 2 lines of the current raw chunk to use as context for the next iteration
        previous_context = '\n'.join(chunk_lines[-2:])

    print("  -> LLM cleanup complete.")

    return "\n\n".join(final_cleaned_chunks)

# Processing

In [6]:
executor = ThreadPoolExecutor(max_workers=2) # For parallel processing

for timeslot_folder in BASE_FOLDER.glob("Timeslot #*"):
    print(f"\n========================================")
    print(f"Processing: {timeslot_folder.name}")

    audio_dir = timeslot_folder / "audio"
    transcript_dir = timeslot_folder / "transcript"
    transcript_dir.mkdir(parents=True, exist_ok=True)

    audio_files = list(audio_dir.glob("*.*"))
    if not audio_files:
        print(f"  -> No audio file found in {audio_dir}. Skipping.")
        continue

    main_audio_path = audio_files[0]
    final_output_path = transcript_dir / f"{timeslot_folder.name}_final.txt"
    raw_output_path = transcript_dir / f"{timeslot_folder.name}_raw_aligned.txt"

    if final_output_path.exists():
        print(f"  -> Final transcript already exists. Skipping.")
        continue

    print(f"  -> Submitting tasks for Parallel Processing...")

    # Run API transcription and local diarization simultaneously
    future_transcription = executor.submit(transcribe_full_audio, main_audio_path)
    future_diarization = executor.submit(diarize_audio_locally, main_audio_path)

    # Wait for both tasks to complete and get their results
    whisper_data = future_transcription.result()
    diarization_data = future_diarization.result()

    # Align text with Speakers
    raw_aligned_text = align_segments(whisper_data, diarization_data)

    # Save the raw aligned text
    with open(raw_output_path, "w", encoding="utf-8") as f:
        f.write(raw_aligned_text)

    # LLM Contextual Cleanup
    final_transcript = clean_transcript_with_llm(raw_aligned_text)

    print(f"  -> Saving final transcript to {final_output_path.name}...")
    with open(final_output_path, "w", encoding="utf-8") as f:
        f.write(f"--- Final Cleaned Transcript for {timeslot_folder.name} ---\n\n")
        f.write(final_transcript)

print("\nAll user study folders processed successfully!")


Processing: Timeslot #1
  -> Final transcript already exists. Skipping.

Processing: Timeslot #10
  -> Final transcript already exists. Skipping.

Processing: Timeslot #11
  -> Final transcript already exists. Skipping.

Processing: Timeslot #12
  -> Final transcript already exists. Skipping.

Processing: Timeslot #13
  -> Final transcript already exists. Skipping.

Processing: Timeslot #14
  -> Submitting tasks for Parallel Processing...
      [API] Compressing audio to meet server upload limits...
      [Local] Starting diarization on cuda...


C:\GitHub\Repositories\TUW-BT\.venv\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
C:\GitHub\Repositories\TUW-BT\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


      [API] Starting full audio transcription...
      [API] Transcription complete.


C:\GitHub\Repositories\TUW-BT\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `float` - serialized value may not be as expected [field_name='duration', input_value='3515.232', input_type=str])
  return self.__pydantic_serializer__.to_python(


      [Local] Diarization complete.
  -> Aligning timestamps...
  -> Sending raw transcript to LLM in chunks for contextual cleanup...
      [LLM] Processing chunk 1/11...
      [LLM] Processing chunk 2/11...
      [LLM] Processing chunk 3/11...
      [LLM] Processing chunk 4/11...
      [LLM] Processing chunk 5/11...
      [LLM] Processing chunk 6/11...
      [LLM] Processing chunk 7/11...
      [LLM] Processing chunk 8/11...
      [LLM] Processing chunk 9/11...
      [LLM] Processing chunk 10/11...
      [LLM] Processing chunk 11/11...
  -> LLM cleanup complete.
  -> Saving final transcript to Timeslot #14_final.txt...

Processing: Timeslot #2
  -> Final transcript already exists. Skipping.

Processing: Timeslot #3
  -> Final transcript already exists. Skipping.

Processing: Timeslot #4
  -> Final transcript already exists. Skipping.

Processing: Timeslot #5
  -> Final transcript already exists. Skipping.

Processing: Timeslot #6
  -> Final transcript already exists. Skipping.

Proce